# Capítulo 6: Sistemas Gênicos
## Processamento, Análise e Bioinformática de Dados Genômicos em Python

Este notebook reúne os exemplos práticos discutidos no Capítulo 6 da disciplina **Bioinformática para Biologia de Sistemas**. Aqui você encontrará as seguintes implementações prontas para executar:
1. **Identificação Computacional de ORFs (Open Reading Frames):** Busca de janelas de leitura em sequência de DNA.
2. **Modelo Matemático de Regulação Gênica:** Implementação e simulação do efeito de cooperatividade (coeficiente de Hill) na taxa de transcrição.
3. **Normalização de Expressão Gênica (TPM):** Conversão de contagens brutas de RNA-seq para Transcripts Per Million (TPM).
4. **Análise de Expressão Diferencial (DEG):** Teste t de Student para identificar genes expressos diferencialmente entre condições e cálculo de log2(Fold Change).
5. **Visualização de Expressão (Volcano Plot):** Geração de Volcano Plot clássico usando matplotlib.
6. **Anotação de Sequências com BioPython:** Leitura de arquivos FASTA e tradução em múltiplas fases (frames).
7. **Acesso Automatizado ao GenBank (NCBI):** Busca e download de registros anotados do GenBank utilizando a biblioteca Entrez do BioPython.
8. **Extração de Características (Features):** Extração estruturada de genes e CDS (Coding Sequences) a partir de registros completos do GenBank.

--- 
## Exemplo 1: Busca Computacional de ORFs (Open Reading Frames)

Uma ORF é uma sequência contínua de DNA que inicia com um códon de início (geralmente ATG) e termina em um códon de parada (TAA, TAG ou TGA) na mesma fase de leitura (frame). O algoritmo abaixo busca por essas regiões em um intervalo de leitura direta.

In [ ]:
def find_orfs(sequence, min_length=100):
    """
    Encontra ORFs em uma sequencia de DNA nas 3 fases de leitura (frames) diretas.
    min_length: comprimento minimo de aminoacidos (padrao: 100 aa = 300 nt)
    """
    start_codon = "ATG"
    stop_codons = ["TAA", "TAG", "TGA"]
    orfs = []

    # Para cada frame (0, 1, 2)
    for frame in range(3):
        for i in range(frame, len(sequence)-2, 3):
            codon = sequence[i:i+3]

            # Encontrou o codon de inicio
            if codon == start_codon:
                # Procurar o proximo stop codon na mesma fase de leitura
                for j in range(i+3, len(sequence)-2, 3):
                    stop = sequence[j:j+3]
                    if stop in stop_codons:
                        orf_len = j - i + 3
                        if orf_len >= min_length * 3:
                            orfs.append({
                                'start': i,
                                'end': j+3,
                                'length': orf_len,
                                'frame': frame
                            })
                        break # Para no primeiro stop codon da fase e busca a proxima ORF
    return orfs

# Sequência de teste sintética
dna_seq = "ATGGCTAAATGGCCATAAGCGATGCCCCCCGGGGGGTAAATGCCTAGCTAGCTAGCTAGTAA"
found = find_orfs(dna_seq, min_length=5)
print(f"ORFs encontradas: {len(found)}")
for idx, orf in enumerate(found):
    print(f"  ORF {idx+1}: Frame {orf['frame']}, Inicio: {orf['start']}, Fim: {orf['end']}, Tamanho: {orf['length']} nt")

--- 
## Exemplo 2: Simulação de Regulação Gênica (Função de Hill)

A regulação da transcrição por fatores de transcrição (TF) cooperativos é frequentemente modelada pela equação de Hill:
$$\text{Taxa} = V_{\max} \frac{[TF]^n}{K_d^n + [TF]^n}$$
onde $n$ representa o coeficiente de Hill (cooperatividade) e $K_d$ representa a afinidade de ligação.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def hill_function(TF, Vmax, Kd, n):
    """Funcao de Hill para regulacao genica cooperativa"""
    return Vmax * (TF**n) / (Kd**n + TF**n)

# Parametros de simulacao
Vmax = 100
Kd = 10
TF_conc = np.linspace(0, 50, 200)

# Plotar efeito de diferentes coeficientes de Hill (cooperatividade)
plt.figure(figsize=(9, 5.5))
for n in [1, 2, 4, 8]:
    mRNA_rate = hill_function(TF_conc, Vmax, Kd, n)
    plt.plot(TF_conc, mRNA_rate, linewidth=2, label=f'n = {n} (cooperatividade)')

# Linhas auxiliares
plt.axhline(y=Vmax/2, color='gray', linestyle='--', alpha=0.7, label='Vmax/2')
plt.axvline(x=Kd, color='gray', linestyle=':', alpha=0.7, label='Kd')

plt.xlabel('Concentração do Fator de Transcrição [TF]')
plt.ylabel('Taxa de Transcrição do mRNA')
plt.title('Regulação Gênica: Efeito da Cooperatividade (Função de Hill)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

--- 
## Exemplo 3: Normalização de Dados de RNA-seq (TPM)

Para comparar a abundância de transcritos, as leituras do sequenciamento (reads) devem ser normalizadas pelo tamanho do gene e pela profundidade de sequenciamento. O método TPM (Transcripts Per Million) é ideal porque garante que a soma das intensidades em cada amostra seja $10^6$.

In [ ]:
import pandas as pd
import numpy as np

def tpm_normalize(counts, gene_lengths):
    """
    Normaliza uma matriz de contagens brutas para TPM (Transcripts Per Million)
    counts: DataFrame do pandas (linhas=genes, colunas=amostras)
    gene_lengths: Série do pandas contendo o tamanho de cada gene em pares de bases (bp)
    """
    # 1. Dividir as contagens pelo tamanho do gene em Kilobases (Reads Per Kilobase - RPK)
    rpk = counts.div(gene_lengths / 1000, axis=0)
    
    # 2. Calcular o fator de escala de profundidade por amostra (soma dos RPKs / 1.000.000)
    scaling_factor = rpk.sum(axis=0) / 1e6
    
    # 3. Dividir o RPK pelo fator de escala
    tpm = rpk.div(scaling_factor, axis=1)
    return tpm

# Gerar dados de contagem sintéticos para teste
np.random.seed(42)
genes = [f"Gene_{i:03d}" for i in range(1, 11)]
samples = ['Control_1', 'Control_2', 'Treat_1', 'Treat_2']
raw_counts = pd.DataFrame(
    np.random.randint(10, 5000, size=(10, 4)), 
    index=genes, 
    columns=samples
)
lengths = pd.Series(np.random.randint(500, 8000, size=10), index=genes)

tpm_counts = tpm_normalize(raw_counts, lengths)
print("Contagens brutas:")
print(raw_counts.head(4))
print("\nValores em TPM (Normalizados):")
print(tpm_counts.head(4))
print("\nSoma de cada amostra normalizada (deve ser 1 milhão):", tpm_counts.sum(axis=0).values)

--- 
## Exemplo 4: Análise de Expressão Diferencial (DEG) e Volcano Plot

Abaixo geramos dados sintéticos correspondentes a 1.000 genes expressos em 3 amostras controle e 3 amostras tratadas, normalizamos, executamos testes t de Student para calcular significância de regulação e plotamos os resultados em um **Volcano Plot** destacando os Genes Diferencialmente Expressos (DEGs).

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# 1. Gerar dados sintéticos maiores para visualização realista
np.random.seed(42)
n_genes = 1000
genes = [f"Gene_{i:04d}" for i in range(1, n_genes + 1)]

# Controle (distribuição normal de log-intensidades)
control_log = np.random.normal(loc=6, scale=1.5, size=(n_genes, 3))
# Tratamento (com variação para simular genes modulados)
treatment_log = control_log.copy()

# Alterar artificialmente alguns genes (modulação)
treatment_log[0:50] += np.random.normal(loc=2.0, scale=0.5, size=(50, 3))   # Super-expressos (UP)
treatment_log[50:100] -= np.random.normal(loc=2.0, scale=0.5, size=(50, 3))  # Sub-expressos (DOWN)
treatment_log[100:] += np.random.normal(loc=0.0, scale=0.2, size=(n_genes-100, 3)) # Sem alteração significativa

# Converter de volta para escala linear (intensidades/contagens)
control_counts = np.exp(control_log)
treatment_counts = np.exp(treatment_log)

# Criar DataFrames
df_control = pd.DataFrame(control_counts, index=genes)
df_treatment = pd.DataFrame(treatment_counts, index=genes)

# 2. Teste t de Student e Cálculo do Log2(Fold Change)
def differential_expression(control, treatment):
    # Teste t de Student independente (gene a gene nas linhas)
    t_stat, p_values = stats.ttest_ind(control, treatment, axis=1)
    
    # Log2 Fold Change (Log2 do quociente das médias)
    mean_control = control.mean(axis=1)
    mean_treatment = treatment.mean(axis=1)
    log_fc = np.log2(mean_treatment / mean_control)
    
    return log_fc, p_values

log_fc, p_vals = differential_expression(df_control, df_treatment)

# 3. Geração do Volcano Plot
neg_log_p = -np.log10(p_vals)

plt.figure(figsize=(10, 8))
# Plotar todos os genes
plt.scatter(log_fc, neg_log_p, alpha=0.4, color='gray', s=15, label='Não significativo')

# Thresholds
p_threshold = 0.05
fc_threshold = 1.0 # Equivalente a 2x mudança (UP/DOWN)
plt.axhline(y=-np.log10(p_threshold), color='black', linestyle=':', alpha=0.7, label=f'Limiar P-value ({p_threshold})')
plt.axvline(x=-fc_threshold, color='blue', linestyle='--', alpha=0.5)
plt.axvline(x=fc_threshold, color='blue', linestyle='--', alpha=0.5, label='Limiar |log2 FC| = 1')

# Identificar e destacar os genes UP-regulados significativos
up_sig = (p_vals < p_threshold) & (log_fc > fc_threshold)
plt.scatter(log_fc[up_sig], neg_log_p[up_sig], color='red', s=25, label='Super-expressos (UP)')

# Identificar e destacar os genes DOWN-regulados significativos
down_sig = (p_vals < p_threshold) & (log_fc < -fc_threshold)
plt.scatter(log_fc[down_sig], neg_log_p[down_sig], color='green', s=25, label='Sub-expressos (DOWN)')

plt.xlabel('Módulo log2(Fold Change)')
plt.ylabel('-log10(P-value)')
plt.title('Volcano Plot: Análise de Expressão Diferencial de Genes')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

--- 
## Exemplo 5: Anotação de Sequências usando BioPython

BioPython oferece pacotes robustos para manipular sequências biológicas. Aqui escrevemos um arquivo FASTA local temporário e o carregamos usando o parser `SeqIO` para traduzir sequências nas 6 possíveis janelas de leitura (frames).

In [ ]:
from Bio import SeqIO
from Bio.SeqUtils import gc_fraction
import os

# Criar um arquivo FASTA temporário local para o teste funcionar autonomamente
with open("sample_genome.fasta", "w") as f:
    f.write(">seq_test_01 Organismo de Teste\n")
    f.write("ATGGCTAAATGGCCATAAGCGATGCCCCCCGGGGGGTAAATGCCTAGCTAGCTAGCTAGTAA\n")

def annotate_sequence(fasta_file):
    """Carrega arquivo FASTA e executa estatisticas e traducao das frames"""
    for record in SeqIO.parse(fasta_file, "fasta"):
        seq = record.seq
        print(f"ID do Registro: {record.id}")
        print(f"Descrição: {record.description}")
        print(f"Tamanho: {len(seq)} pb")
        # Nota: gc_fraction é o substituto moderno para GC no BioPython
        print(f"Conteúdo GC: {gc_fraction(seq)*100:.2f}%")
        
        print("\nORFs Traduzidas detectadas (Fitas Direta e Reversa):")
        # Traduzir as 6 frames (+1, +2, +3 e -1, -2, -3)
        strands = [(+1, seq), (-1, seq.reverse_complement())]
        for direction, nucleotide_seq in strands:
            for frame in range(3):
                # Cortar para tamanho multiplo de 3
                length = 3 * ((len(nucleotide_seq) - frame) // 3)
                translated = nucleotide_seq[frame:frame+length].translate()
                
                # Buscar trechos entre codons de parada (*)
                peptides = translated.split("*")
                for peptide in peptides:
                    if len(peptide) >= 5: # Filtro simples de tamanho para impressao
                        print(f"  Fita {direction:+} | Frame {frame+1} | Peptídeo ({len(peptide)} aa): {peptide}")

# Executar
annotate_sequence("sample_genome.fasta")

# Limpar arquivo temporário
if os.path.exists("sample_genome.fasta"):
    os.remove("sample_genome.fasta")

--- 
## Exemplo 6: Acesso ao GenBank (NCBI) e Extração de Features com BioPython

Podemos fazer downloads de genomas e RNAs anotados diretamente do NCBI via API `Entrez` e extrair as anotações do registro (`features` como CDS e genes).

In [ ]:
from Bio import Entrez, SeqIO

# O NCBI exige o fornecimento de um e-mail para identificação nas requisições do Entrez
Entrez.email = "seu_email@example.com"

def fetch_and_parse_genbank(accession_id):
    """
    Baixa um registro completo em formato GenBank do NCBI e extrai features.
    """
    print(f"Baixando registro {accession_id} do NCBI...")
    try:
        # Fazer download usando efetch
        handle = Entrez.efetch(
            db="nucleotide",
            id=accession_id,
            rettype="gb",
            retmode="text"
        )
        record = SeqIO.read(handle, "genbank")
        handle.close()
        
        print(f"ID: {record.id}")
        print(f"Definição: {record.description}")
        print(f"Tamanho: {len(record.seq)} pb")
        print(f"Organismo: {record.annotations.get('organism', 'Desconhecido')}")
        
        # Contadores de features
        gene_count = 0
        cds_count = 0
        
        print("\nResumo das CDS encontradas no registro:")
        for feature in record.features:
            if feature.type == "gene":
                gene_count += 1
            elif feature.type == "CDS":
                cds_count += 1
                product = feature.qualifiers.get("product", ["Unknown"])[0]
                gene_sym = feature.qualifiers.get("gene", ["?"])[0]
                start = feature.location.start
                end = feature.location.end
                print(f"  Gene: {gene_sym:5} | Produto: {product[:35]:35} | Localização: {start}-{end} ({feature.location.strand:+})")
                
        print(f"\nTotal de Genes: {gene_count} | Total de Regiões Codificantes (CDS): {cds_count}")
        return record
        
    except Exception as e:
        print(f"Ocorreu um erro no download ou parser: {e}")
        print("Dica: Certifique-se de que possui conexao com a internet ativa.")
        return None

# Exemplo: Baixar o mRNA da beta-hemoglobina humana (HBB) - Acesso: NM_000518
hbb_record = fetch_and_parse_genbank("NM_000518")